# ⚡ Google Colab Cloud GPU Worker for Bio-Quant Engine
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
### High-Performance Nvidia T4 / L4 / A100 Accelerator (High-RAM)

This notebook executes GPU-accelerated **PyTorch CUDA Tensor AST Evolution** and **MAP-Elites Search** for quantitative alphas.

> **Colab GPU Setup Instructions**:
> 1. Click **Runtime** > **Change runtime type**
> 2. Select **T4 GPU** (or A100 if Pro) and **High-RAM**
> 3. Run all cells sequentially

#### Two Operational Workflows:
1. **Autonomous Search & JSON Export (Recommended)**:
   - Run Cells 1 to 4 to search for alpha champions directly on GPU.
   - Exports champions to `discovered_elites_{ticker}.json`.
   - Download the JSON file and import into sovereign database with:
     `python trade_new/scripts/import_cloud_elites.py --file discovered_elites_BANKBEES.json --ticker BANKBEES`
2. **Live Cloud GPU Tunnel (Cockpit Integration)**:
   - Run Cell 5 to start FastAPI server and Cloudflare Tunnel.
   - Stream evolution directly to your local Bio-Quant Cockpit WebUI (http://127.0.0.1:7860).


In [ ]:
# ==============================================================================
# 1. DEPENDENCY SETUP & CUDA HARDWARE DETECTION
# ==============================================================================
import os
import sys
import gc
import ast
import json
import time
import math
import random
import threading
import re
import subprocess
from typing import List, Dict, Any, Optional
import numpy as np
import pandas as pd

print("[1/5] Checking and installing cloud dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "yfinance", "loguru", "pydantic", "requests", "psutil", "scipy", "torch"], check=False)
if sys.platform != "win32":
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=False)
    subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], check=False)

import torch
import torch.nn.functional as F
import requests
import uvicorn
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_GPU = torch.cuda.is_available()
DEVICE_NAME = torch.cuda.get_device_name(0) if USE_GPU else "CPU Host"
VRAM_TOTAL_GB = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if USE_GPU else 0.0

import psutil
RAM_TOTAL_GB = round(psutil.virtual_memory().total / (1024**3), 2)

class HardwareArenaManager:
    """Guards against OOM crashes by monitoring VRAM and triggering aggressive GC."""
    @staticmethod
    def telemetry() -> Dict[str, Any]:
        mem = psutil.virtual_memory()
        return {
            "device": DEVICE_NAME,
            "gpu_acceleration": USE_GPU,
            "vram_total_gb": VRAM_TOTAL_GB,
            "vram_used_gb": round(torch.cuda.memory_allocated(0) / (1024**3), 2) if USE_GPU else 0.0,
            "ram_total_gb": RAM_TOTAL_GB,
            "ram_used_gb": round(mem.used / (1024**3), 2),
            "ram_percent": mem.percent
        }

    @staticmethod
    def guard_headroom(vram_threshold_pct: float = 0.88):
        if USE_GPU:
            used = torch.cuda.memory_allocated(0)
            total = torch.cuda.get_device_properties(0).total_memory
            if total > 0 and (used / total) > vram_threshold_pct:
                torch.cuda.empty_cache()
                gc.collect()

print(f"[2/5] Hardware active: {DEVICE_NAME} | VRAM: {VRAM_TOTAL_GB} GB | RAM: {RAM_TOTAL_GB} GB | GPU Acceleration: {USE_GPU}")


In [ ]:
# ==============================================================================
# 2. PYTORCH CUDA TENSOR AST EVALUATOR & MAP-ELITES ENGINE
# ==============================================================================
OPERATORS_BINARY = ("ADD", "SUB", "MUL", "DIV_SAFE")
OPERATORS_UNARY = ("SIGN", "ABS", "LOG_ABS", "SIGMOID", "RELU")
OPERATORS_ROLLING = ("DELTA_5", "EMA_10", "TS_MAX_10", "TS_MIN_10", "TS_RANK_20")
ALL_OPERATORS = OPERATORS_BINARY + OPERATORS_UNARY + OPERATORS_ROLLING

STANDARD_FEATURES = [
    "Close_Adj", "Open", "High", "Low", "Volume", "Spread",
    "Vol_ZScore", "Vol_Price_Flow", "True_Range", "TR_Expansion",
    "EMA_Slope", "Dist_EMA20", "Dist_VWAP", "Downside_Vol_Ratio",
    "RSI_14", "Parkinson_Vol", "Garman_Klass_Vol",
    "SP500_Ret", "ADR_Ret", "Global_Confluence"
]

EMA10_ALPHA = 2.0 / 11.0
EMA10_TAPS = 64
_ema_weights = np.array([EMA10_ALPHA * ((1.0 - EMA10_ALPHA) ** k) for k in range(EMA10_TAPS)], dtype=np.float32)
EMA10_KERNEL = torch.tensor(_ema_weights[::-1].copy(), device=DEVICE, dtype=torch.float32).view(1, 1, EMA10_TAPS)

class ASTNode:
    __slots__ = ("op", "val", "active_regimes", "children", "_hash_str")
    def __init__(self, op: str, val: str = "", active_regimes=None, children=None):
        self.op = op
        self.val = val
        self.active_regimes = active_regimes if active_regimes is not None else set(range(10))
        self.children = children if children is not None else []
        self._hash_str = None

    def clone(self) -> "ASTNode":
        return ASTNode(self.op, self.val, set(self.active_regimes), [c.clone() for c in self.children])

    def to_formula_str(self) -> str:
        if self._hash_str is not None:
            return self._hash_str
        if self.op == "FEATURE":
            res = self.val
        elif self.op in (OPERATORS_ROLLING + OPERATORS_UNARY):
            child = self.children[0].to_formula_str() if self.children else "Close_Adj"
            res = f"{self.op}({child})"
        elif self.op in OPERATORS_BINARY:
            sym = {"ADD": "+", "SUB": "-", "MUL": "*", "DIV_SAFE": "/"}.get(self.op, self.op)
            l = self.children[0].to_formula_str() if self.children else "0"
            r = self.children[1].to_formula_str() if len(self.children) > 1 else "0"
            res = f"({l} {sym} {r})"
        else:
            res = f"{self.op}({self.val})"
        self._hash_str = res
        return res

class FormulaParser:
    BIN_MAP = {ast.Add: "ADD", ast.Sub: "SUB", ast.Mult: "MUL", ast.Div: "DIV_SAFE"}

    @classmethod
    def parse(cls, expr_str: str) -> Optional[ASTNode]:
        clean = expr_str.strip()
        if not clean or clean.startswith("#"):
            return None
        try:
            tree = ast.parse(clean, mode="eval")
            return cls._convert(tree.body)
        except Exception:
            return None

    @classmethod
    def _convert(cls, node) -> ASTNode:
        if isinstance(node, ast.BinOp) and type(node.op) in cls.BIN_MAP:
            return ASTNode(op=cls.BIN_MAP[type(node.op)], children=[cls._convert(node.left), cls._convert(node.right)])
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            return ASTNode(op="SUB", children=[ASTNode(op="FEATURE", val="0.0"), cls._convert(node.operand)])
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            fn = node.func.id.upper()
            if fn in (OPERATORS_UNARY + OPERATORS_ROLLING):
                arg = cls._convert(node.args[0]) if node.args else ASTNode(op="FEATURE", val="Close_Adj")
                return ASTNode(op=fn, children=[arg])
            if fn in OPERATORS_BINARY and len(node.args) >= 2:
                return ASTNode(op=fn, children=[cls._convert(node.args[0]), cls._convert(node.args[1])])
        if isinstance(node, ast.Name):
            return ASTNode(op="FEATURE", val=node.id)
        if isinstance(node, ast.Constant):
            return ASTNode(op="FEATURE", val=str(node.value))
        return ASTNode(op="FEATURE", val="Close_Adj")

class TensorASTEvaluator:
    _eval_cache: Dict[str, torch.Tensor] = {}

    @classmethod
    def clear_cache(cls):
        cls._eval_cache.clear()

    @staticmethod
    def eval_node(node: ASTNode, feature_tensors: Dict[str, torch.Tensor], n_len: int, regime: int = 0) -> torch.Tensor:
        node_key = node.to_formula_str()
        if node_key in TensorASTEvaluator._eval_cache:
            return TensorASTEvaluator._eval_cache[node_key]

        if regime != -1 and regime not in node.active_regimes:
            res = torch.zeros(n_len, device=DEVICE, dtype=torch.float32)
            TensorASTEvaluator._eval_cache[node_key] = res
            return res

        if node.op == "FEATURE":
            if node.val in feature_tensors:
                res = feature_tensors[node.val]
            else:
                try:
                    res = torch.full((n_len,), float(node.val), device=DEVICE, dtype=torch.float32)
                except ValueError:
                    res = torch.zeros(n_len, device=DEVICE, dtype=torch.float32)
            TensorASTEvaluator._eval_cache[node_key] = res
            return res

        if node.op in OPERATORS_BINARY:
            l = TensorASTEvaluator.eval_node(node.children[0], feature_tensors, n_len, regime) if len(node.children) > 0 else torch.zeros(n_len, device=DEVICE)
            r = TensorASTEvaluator.eval_node(node.children[1], feature_tensors, n_len, regime) if len(node.children) > 1 else torch.zeros(n_len, device=DEVICE)
            if node.op == "ADD": res = l + r
            elif node.op == "SUB": res = l - r
            elif node.op == "MUL": res = l * r
            elif node.op == "DIV_SAFE": res = torch.where(torch.abs(r) > 1e-6, l / (r + 1e-8), torch.zeros_like(l))
            else: res = torch.zeros(n_len, device=DEVICE)
            res = torch.nan_to_num(res, nan=0.0, posinf=0.0, neginf=0.0)
            TensorASTEvaluator._eval_cache[node_key] = res
            return res

        elif node.op in OPERATORS_UNARY:
            c = TensorASTEvaluator.eval_node(node.children[0], feature_tensors, n_len, regime) if node.children else torch.zeros(n_len, device=DEVICE)
            if node.op == "SIGN": res = torch.sign(c)
            elif node.op == "ABS": res = torch.abs(c)
            elif node.op == "LOG_ABS": res = torch.log1p(torch.abs(c))
            elif node.op == "SIGMOID": res = torch.sigmoid(torch.clamp(c, -10.0, 10.0))
            elif node.op == "RELU": res = torch.relu(c)
            else: res = torch.zeros(n_len, device=DEVICE)
            res = torch.nan_to_num(res, nan=0.0, posinf=0.0, neginf=0.0)
            TensorASTEvaluator._eval_cache[node_key] = res
            return res

        elif node.op in OPERATORS_ROLLING:
            c = TensorASTEvaluator.eval_node(node.children[0], feature_tensors, n_len, regime) if node.children else torch.zeros(n_len, device=DEVICE)
            if node.op == "DELTA_5":
                res = torch.zeros_like(c)
                res[5:] = c[5:] - c[:-5]
            elif node.op == "EMA_10":
                pad_c = F.pad(c.view(1, 1, -1), (EMA10_TAPS - 1, 0), mode="replicate")
                res = F.conv1d(pad_c, EMA10_KERNEL).view(-1)[:n_len]
            elif node.op == "TS_MAX_10":
                pad_c = F.pad(c.view(1, 1, -1), (9, 0), mode="replicate")
                res = F.max_pool1d(pad_c, kernel_size=10, stride=1).view(-1)[:n_len]
            elif node.op == "TS_MIN_10":
                pad_c = F.pad(-c.view(1, 1, -1), (9, 0), mode="replicate")
                res = -F.max_pool1d(pad_c, kernel_size=10, stride=1).view(-1)[:n_len]
            elif node.op == "TS_RANK_20":
                pad_c = F.pad(c.view(1, 1, -1), (19, 0), mode="replicate").view(-1)
                windows = pad_c.unfold(0, 20, 1)
                res = ((windows < windows[:, -1:]).sum(dim=-1).float() + 0.5 * (windows == windows[:, -1:]).sum(dim=-1).float()) / 20.0
            else:
                res = torch.zeros(n_len, device=DEVICE)
            res = torch.nan_to_num(res, nan=0.0, posinf=0.0, neginf=0.0)
            TensorASTEvaluator._eval_cache[node_key] = res
            return res

        res = torch.zeros(n_len, device=DEVICE, dtype=torch.float32)
        TensorASTEvaluator._eval_cache[node_key] = res
        return res

class GeneticEngine:
    @staticmethod
    def generate(max_depth: int = 3, features: List[str] = STANDARD_FEATURES, active_regimes=None) -> ASTNode:
        regimes = active_regimes if active_regimes is not None else set(range(10))
        if max_depth <= 1 or random.random() < 0.30:
            if random.random() < 0.70:
                return ASTNode(op="FEATURE", val=random.choice(features), active_regimes=regimes)
            return ASTNode(op="FEATURE", val=str(round(random.uniform(-2.0, 2.0), 2)), active_regimes=regimes)

        op = random.choice(ALL_OPERATORS)
        if op in OPERATORS_BINARY:
            return ASTNode(op=op, active_regimes=regimes, children=[
                GeneticEngine.generate(max_depth - 1, features, regimes),
                GeneticEngine.generate(max_depth - 1, features, regimes)
            ])
        return ASTNode(op=op, active_regimes=regimes, children=[GeneticEngine.generate(max_depth - 1, features, regimes)])

    @staticmethod
    def mutate(node: ASTNode, max_depth: int = 3, features: List[str] = STANDARD_FEATURES) -> ASTNode:
        if random.random() < 0.35:
            return GeneticEngine.generate(max_depth, features, node.active_regimes)
        cloned = node.clone()
        if cloned.children:
            idx = random.randint(0, len(cloned.children) - 1)
            cloned.children[idx] = GeneticEngine.mutate(cloned.children[idx], max_depth - 1, features)
        return cloned

    @staticmethod
    def crossover(p1: ASTNode, p2: ASTNode, max_depth: int = 3) -> tuple:
        c1, c2 = p1.clone(), p2.clone()
        if c1.children and c2.children:
            i1 = random.randint(0, len(c1.children) - 1)
            i2 = random.randint(0, len(c2.children) - 1)
            c1.children[i1], c2.children[i2] = c2.children[i2], c1.children[i1]
        return c1, c2

    @staticmethod
    def warm_start_seeds() -> List[ASTNode]:
        seeds = [
            "(RSI_14 - 0.50)",
            "EMA_10(Vol_Price_Flow)",
            "((Close_Adj - Dist_VWAP) / (Parkinson_Vol + 0.01))",
            "(TS_RANK_20(Close_Adj) - 0.50)",
            "(EMA_Slope - Dist_EMA20)",
            "TS_MAX_10(Vol_ZScore)",
            "(Downside_Vol_Ratio * RSI_14)",
            "(0.0 - Dist_VWAP)"
        ]
        res = []
        for s in seeds:
            parsed = FormulaParser.parse(s)
            if parsed:
                res.append(parsed)
        return res

class EliteIndividual:
    def __init__(self, formula: ASTNode, fitness: float, win_rate: float, net_return_pct: float,
                 turnover: float, volatility: float, sortino: float, profit_factor: float,
                 trades_count: int, max_drawdown: float, cagr_pct: float, lookback: str = "5y"):
        self.formula = formula
        self.fitness = fitness
        self.win_rate = win_rate
        self.net_return_pct = net_return_pct
        self.turnover = turnover
        self.volatility = volatility
        self.sortino = sortino
        self.profit_factor = profit_factor
        self.trades_count = trades_count
        self.max_drawdown = max_drawdown
        self.cagr_pct = cagr_pct
        self.lookback = lookback

class MapElitesArchive:
    def __init__(self, rows: int = 20, cols: int = 20):
        self.rows, self.cols = rows, cols
        self.grid = np.empty((rows, cols), dtype=object)
        self.vol_bins = np.linspace(0.05, 0.60, rows)
        self.turnover_bins = np.linspace(2.0, 150.0, cols)

    def evaluate_batch(self, batch_nodes: List[ASTNode], feature_tensors: Dict[str, torch.Tensor],
                       gross_rets_t: torch.Tensor, n_len: int, friction: float = 0.00012, target_mode: str = "overnight") -> int:
        if not batch_nodes:
            return 0
        HardwareArenaManager.guard_headroom()
        inserted = 0
        with torch.no_grad():
            sig_list = [TensorASTEvaluator.eval_node(node, feature_tensors, n_len) for node in batch_nodes]
            sigs = torch.stack(sig_list, dim=0)
            pos = (sigs > 0.0).float()
            trade_pos = pos[:, :-1]
            eff_rets = gross_rets_t[:-1] if len(gross_rets_t) > trade_pos.shape[1] else gross_rets_t

            B_valid, T_sub = trade_pos.shape
            zero_col = torch.zeros((B_valid, 1), device=DEVICE, dtype=torch.float32)
            flips = torch.abs(torch.diff(trade_pos, dim=1, prepend=zero_col))
            net_rets = (trade_pos * eff_rets.unsqueeze(0)) - (flips * friction)

            entries = (flips > 0.0) & (trade_pos > 0.0)
            n_entries = entries.sum(dim=1)
            valid_mask = n_entries >= 5
            if not torch.any(valid_mask):
                return 0

            valid_idx = torch.nonzero(valid_mask, as_tuple=False).view(-1)
            trade_pos = trade_pos[valid_idx]
            net_rets = net_rets[valid_idx]
            flips = flips[valid_idx]
            n_entries = n_entries[valid_idx]

            active_trades = trade_pos > 0.0
            cum_eq = torch.cumprod(1.0 + net_rets, dim=1)
            tot_ret_vec = cum_eq[:, -1] - 1.0

            peaks = torch.cummax(cum_eq, dim=1).values
            dd = (peaks - cum_eq) / (peaks + 1e-8)
            max_dd_vec = torch.max(dd, dim=1).values * 100.0

            vol_vec = torch.std(net_rets, dim=1) * math.sqrt(252.0)
            turnover_vec = torch.mean(flips, dim=1) * 252.0

            active_rets = torch.where(active_trades, net_rets, torch.zeros_like(net_rets))
            n_active = torch.clamp(active_trades.sum(dim=1).float(), min=1.0)
            mean_ret_vec = active_rets.sum(dim=1) / n_active
            downside_rets = torch.clamp(active_rets, max=0.0)
            downside_dev = torch.clamp(torch.sqrt((downside_rets ** 2).sum(dim=1) / n_active), min=0.002)
            sortino_vec = (mean_ret_vec / downside_dev) * math.sqrt(252.0)

            gross_gains = torch.where(net_rets > 0.0, net_rets, torch.zeros_like(net_rets)).sum(dim=1)
            gross_losses = torch.clamp(torch.abs(torch.where(net_rets < 0.0, net_rets, torch.zeros_like(net_rets)).sum(dim=1)), min=0.0001)
            pf_vec = gross_gains / gross_losses

            wins_vec = ((net_rets > 0.0) & active_trades).float().sum(dim=1)
            win_rate_vec = (wins_vec / n_active) * 100.0

            years = max(0.08, n_len / 252.0)
            cagr_vec = torch.where(
                tot_ret_vec > -0.99,
                (torch.pow(torch.clamp(1.0 + tot_ret_vec, min=0.001), 1.0 / years) - 1.0) * 100.0,
                torch.full_like(tot_ret_vec, -100.0)
            )

            calmar_ratio = torch.clamp(cagr_vec, min=0.0) / torch.clamp(max_dd_vec, min=4.0)
            expected_trades = max(15.0, 5.0 * years)
            trade_conf = torch.clamp(torch.sqrt(n_entries.float() / expected_trades), min=0.2, max=1.2)
            quality_mult = torch.clamp(torch.sqrt(torch.clamp(sortino_vec, min=0.25, max=16.0)), min=0.5, max=3.0)
            t_pen = torch.where(turnover_vec > 180.0, torch.clamp(180.0 / (turnover_vec + 1e-5), min=0.1), torch.ones_like(turnover_vec))
            fitness_vec = calmar_ratio * trade_conf * quality_mult * t_pen

            for b_idx, cand_idx in enumerate(valid_idx.tolist()):
                vol = float(vol_vec[b_idx].item())
                turn = float(turnover_vec[b_idx].item())
                fit = float(fitness_vec[b_idx].item())
                if math.isnan(fit) or math.isinf(fit) or fit <= 0.0:
                    continue

                r_bin = int(np.clip(np.digitize(vol, self.vol_bins) - 1, 0, self.rows - 1))
                c_bin = int(np.clip(np.digitize(turn, self.turnover_bins) - 1, 0, self.cols - 1))

                curr = self.grid[r_bin, c_bin]
                if curr is None or fit > curr.fitness:
                    self.grid[r_bin, c_bin] = EliteIndividual(
                        formula=batch_nodes[cand_idx],
                        fitness=fit,
                        win_rate=float(win_rate_vec[b_idx].item()),
                        net_return_pct=float(tot_ret_vec[b_idx].item() * 100.0),
                        turnover=turn,
                        volatility=vol,
                        sortino=float(sortino_vec[b_idx].item()),
                        profit_factor=float(pf_vec[b_idx].item()),
                        trades_count=int(n_entries[b_idx].item()),
                        max_drawdown=float(max_dd_vec[b_idx].item()),
                        cagr_pct=float(cagr_vec[b_idx].item())
                    )
                    inserted += 1
        return inserted

    def get_residents(self) -> List[EliteIndividual]:
        return [c for c in self.grid.flatten() if c is not None]

    def get_top_elites(self, n: int = 10) -> List[EliteIndividual]:
        return sorted(self.get_residents(), key=lambda x: x.fitness, reverse=True)[:n]

    def get_all_records(self) -> List[Dict[str, Any]]:
        records = []
        for r in range(self.rows):
            for c in range(self.cols):
                e = self.grid[r, c]
                if e is not None:
                    records.append({
                        "row": r, "col": c, "fitness": round(float(e.fitness), 4),
                        "sortino": round(float(e.sortino), 4),
                        "net_return_pct": round(float(e.net_return_pct), 2),
                        "cagr_pct": round(float(getattr(e, "cagr_pct", e.net_return_pct)), 2),
                        "max_drawdown": round(float(e.max_drawdown), 4),
                        "win_rate": round(float(e.win_rate), 2),
                        "profit_factor": round(float(e.profit_factor), 2),
                        "turnover": round(float(e.turnover), 2),
                        "volatility": round(float(e.volatility), 4),
                        "trades_count": int(e.trades_count),
                        "formula_str": e.formula.to_formula_str(),
                        "active_regimes": list(e.formula.active_regimes)
                    })
        return records

DATA_CACHE = {}

def get_or_fetch_market_buffers(ticker: str, lookback: str = "5y", target_mode: str = "overnight") -> Dict[str, Any]:
    clean_ticker = ticker.replace(".NS", "").replace(".BO", "")
    cache_key = f"{clean_ticker}_{lookback}_{target_mode}"
    if cache_key in DATA_CACHE:
        return DATA_CACHE[cache_key]

    df = None
    try:
        import yfinance as yf
        for sym in [ticker, f"{clean_ticker}.NS", f"{clean_ticker}.BO", clean_ticker]:
            try:
                raw = yf.download(sym, period=lookback, progress=False, timeout=12, auto_adjust=False)
                if raw is not None and len(raw) >= 50:
                    if isinstance(raw.columns, pd.MultiIndex):
                        raw.columns = [c[0] for c in raw.columns]
                    df = raw.copy()
                    break
            except Exception:
                pass
    except Exception:
        pass

    if df is None or len(df) < 50:
        dates = pd.date_range("2022-01-01", periods=1000, freq="B")
        n = len(dates)
        np.random.seed(42)
        daily_rets = np.random.normal(0.0005, 0.012, n)
        p = 250.0 * np.cumprod(1.0 + daily_rets)
        h = p * (1.0 + np.abs(np.random.normal(0.004, 0.003, n)))
        l = p * (1.0 - np.abs(np.random.normal(0.004, 0.003, n)))
        o = l + (h - l) * np.random.uniform(0.2, 0.8, n)
        v = np.random.lognormal(14.0, 0.5, n)
        df = pd.DataFrame({"Close_Adj": p, "Close_Raw": p, "Open": o, "High": h, "Low": l, "Volume": v}, index=dates)

    if "Close_Adj" not in df.columns:
        df["Close_Adj"] = df.get("Adj Close", df.get("Close", df["Open"]))
    c = df["Close_Adj"]
    h = df.get("High", c)
    l = df.get("Low", c)
    o = df.get("Open", c)
    v = df.get("Volume", pd.Series(1e6, index=df.index))

    df["Spread"] = (h - l) / (c + 1e-8)
    v_ma = v.rolling(20, min_periods=1).mean()
    v_std = v.rolling(20, min_periods=1).std().replace(0, 1.0)
    df["Vol_ZScore"] = (v - v_ma) / v_std
    rets = c.pct_change().fillna(0.0)
    df["Vol_Price_Flow"] = df["Vol_ZScore"] * rets
    tr1, tr2, tr3 = h - l, (h - c.shift(1)).abs().fillna(0.0), (l - c.shift(1)).abs().fillna(0.0)
    df["True_Range"] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1) / (c + 1e-8)
    atr20 = df["True_Range"].rolling(20, min_periods=1).mean()
    df["TR_Expansion"] = df["True_Range"] / (atr20 + 1e-8)
    ema20 = c.ewm(span=20, adjust=False).mean()
    first_ema = float(ema20.iloc[0])
    df["EMA_Slope"] = (ema20 - ema20.shift(5).fillna(first_ema)) / (ema20.shift(5).fillna(first_ema) + 1e-8)
    df["Dist_EMA20"] = (c - ema20) / (ema20 + 1e-8)
    pv = (c * v).cumsum()
    vv = v.cumsum()
    vwap = pv / (vv + 1e-8)
    df["Dist_VWAP"] = (c - vwap) / (vwap + 1e-8)
    down_rets = rets.where(rets < 0, 0.0)
    df["Downside_Vol_Ratio"] = np.sqrt(down_rets.rolling(20, min_periods=1).var() / (rets.rolling(20, min_periods=1).var() + 1e-8)).fillna(0.5)
    delta = c.diff().fillna(0.0)
    gain = (delta.where(delta > 0, 0.0)).rolling(14, min_periods=1).mean()
    loss = (-delta.where(delta < 0, 0.0)).rolling(14, min_periods=1).mean()
    df["RSI_14"] = (100.0 - (100.0 / (1.0 + (gain / (loss + 1e-8))))) / 100.0
    log_hl = np.log(h / (l + 1e-8)) ** 2
    log_co = np.log(c / (o + 1e-8)) ** 2
    df["Parkinson_Vol"] = np.sqrt(log_hl.rolling(20, min_periods=1).mean() / (4.0 * np.log(2.0))) * np.sqrt(252)
    df["Garman_Klass_Vol"] = np.sqrt((0.5 * log_hl - (2.0 * np.log(2.0) - 1.0) * log_co).rolling(20, min_periods=1).mean()) * np.sqrt(252)
    df["SP500_Ret"] = 0.0004
    df["ADR_Ret"] = 0.0005
    df["Global_Confluence"] = 1.0
    df = df.fillna(0.0)

    n_len = len(df)
    c_np, o_np = df["Close_Adj"].to_numpy(dtype=np.float32), df["Open"].to_numpy(dtype=np.float32)
    gross_rets = (o_np[1:] - c_np[:-1]) / (c_np[:-1] + 1e-8)
    gross_rets = np.clip(gross_rets, -0.10, 0.10).astype(np.float32)
    feature_tensors = {col: torch.tensor(df[col].to_numpy(dtype=np.float32), device=DEVICE) for col in df.columns if col in STANDARD_FEATURES}
    gross_rets_t = torch.tensor(gross_rets, device=DEVICE)

    bundle = {
        "n_len": n_len,
        "feature_tensors": feature_tensors,
        "gross_rets_t": gross_rets_t,
        "df": df
    }
    DATA_CACHE[cache_key] = bundle
    return bundle

print("✅ TensorASTEvaluator and MAP-Elites Engine compiled on", DEVICE_NAME)


In [ ]:
# ==============================================================================
# 3. AUTONOMOUS MAP-ELITES GPU SEARCH & DISCOVERY RUNNER
# ==============================================================================
def run_map_elites_evolution(
    ticker: str = "BANKBEES.NS",
    lookback: str = "5y",
    generations: int = 20,
    population_size: int = 100,
    target_mode: str = "overnight",
    chunk_size: int = 256
) -> MapElitesArchive:
    """Runs autonomous MAP-Elites search on PyTorch GPU accelerator."""
    print(f"🚀 Starting Evolution on {DEVICE_NAME} for {ticker} (Lookback: {lookback}, Target: {target_mode})...")
    bundle = get_or_fetch_market_buffers(ticker, lookback=lookback, target_mode=target_mode)
    n_len = bundle["n_len"]
    friction = 0.00012 if "BEES" in ticker else 0.00030

    archive = MapElitesArchive(rows=20, cols=20)
    pop: List[ASTNode] = [GeneticEngine.generate(max_depth=3) for _ in range(population_size)]
    pop.extend(GeneticEngine.warm_start_seeds())

    start_t = time.time()
    for gen in range(1, generations + 1):
        residents = archive.get_residents()
        parent_pool = [e.formula for e in residents] if len(residents) >= 4 else pop
        offspring: List[ASTNode] = []

        while len(offspring) < population_size:
            p1, p2 = random.choice(parent_pool), random.choice(parent_pool)
            if random.random() < 0.60:
                c1, c2 = GeneticEngine.crossover(p1, p2, max_depth=4)
            else:
                c1, c2 = p1.clone(), p2.clone()
            if random.random() < 0.30:
                c1 = GeneticEngine.mutate(c1, max_depth=4)
            if random.random() < 0.30:
                c2 = GeneticEngine.mutate(c2, max_depth=4)
            offspring.extend([c1, c2])

        offspring = offspring[:population_size]
        TensorASTEvaluator.clear_cache()
        effective_chunk = min(max(chunk_size, 1), 1024 if USE_GPU else 256)
        inserted = 0
        for i in range(0, len(offspring), effective_chunk):
            batch_slice = offspring[i : i + effective_chunk]
            inserted += archive.evaluate_batch(
                batch_slice,
                bundle["feature_tensors"],
                bundle["gross_rets_t"],
                n_len,
                friction=friction,
                target_mode=target_mode
            )

        pop = offspring
        residents = archive.get_residents()
        top10 = archive.get_top_elites(10)
        best_calmar = top10[0].fitness if top10 else 0.0
        best_cagr = getattr(top10[0], "cagr_pct", 0.0) if top10 else 0.0
        best_sortino = getattr(top10[0], "sortino", 0.0) if top10 else 0.0
        best_dd = getattr(top10[0], "max_drawdown", 0.0) if top10 else 0.0
        occ = (len(residents) / 400.0) * 100.0
        print(f"[Gen {gen:02d}/{generations:02d}] Residents: {len(residents):3d}/400 ({occ:4.1f}%) | Inserted: {inserted:2d} | Calmar: {best_calmar:5.2f} | Sortino: {best_sortino:5.2f} | CAGR: +{best_cagr:5.1f}% | MaxDD: {best_dd:4.1f}%")

    elapsed = time.time() - start_t
    print(f"\n✅ Evolution Completed in {elapsed:.2f}s! Discovered {len(archive.get_residents())} distinct elite alphas.")
    return archive

# --- Execute Evolution ---
TARGET_TICKER = "BANKBEES.NS"
LOOKBACK_WINDOW = "5y"
active_archive = run_map_elites_evolution(ticker=TARGET_TICKER, lookback=LOOKBACK_WINDOW, generations=15, population_size=100)


In [ ]:
# ==============================================================================
# 4. EXPORT DISCOVERED CHAMPIONS TO CLEAN JSON FILE
# ==============================================================================
def export_discovered_elites(
    archive: MapElitesArchive,
    ticker: str = TARGET_TICKER,
    lookback: str = LOOKBACK_WINDOW,
    output_dir: str = "."
) -> str:
    """Exports discovered champions to clean JSON for sovereign audit & database import."""
    clean_t = ticker.replace(".NS", "").replace(".BO", "")
    export_filename = os.path.join(output_dir, f"discovered_elites_{clean_t}.json")
    records = archive.get_all_records()

    payload = {
        "ticker": clean_t,
        "lookback": lookback,
        "exported_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        "total_elites": len(records),
        "elites": records
    }

    with open(export_filename, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    abs_path = os.path.abspath(export_filename)
    print("=" * 82)
    print("💎 DISCOVERED ELITES EXPORTED SUCCESSFULLY")
    print(f"   Ticker        : {clean_t}")
    print(f"   Lookback      : {lookback}")
    print(f"   Total Elites  : {len(records)}")
    print(f"   File Path     : {abs_path}")
    print("=" * 82)
    print("\nTo audit & import these into your local database, run:")
    print(f"   python trade_new/scripts/import_cloud_elites.py --file discovered_elites_{clean_t}.json --ticker {clean_t}")
    print("=" * 82 + "\n")
    return export_filename

export_filename = export_discovered_elites(active_archive, ticker=TARGET_TICKER, lookback=LOOKBACK_WINDOW)
abs_path = os.path.abspath(export_filename)

# Automatically trigger Colab browser file download
try:
    from google.colab import files
    files.download(export_filename)
    print(f"📥 Colab file download initiated for: {export_filename}")
except Exception as dl_err:
    print(f"Colab download notice: {dl_err}. File saved on disk at: {abs_path}")


In [ ]:
# ==============================================================================
# 5. (OPTIONAL) CLOUD GPU BRIDGE: FASTAPI SERVER & CLOUDFLARE TUNNEL
#    Run this cell only if you want to stream directly to your local WebUI.
# ==============================================================================
app = FastAPI(title="Bio-Quant Cloud GPU Worker")
ACTIVE_ARCHIVES: Dict[str, MapElitesArchive] = {}

class EvolutionJobRequest(BaseModel):
    ticker: str = "BANKBEES.NS"
    lookback: str = "5y"
    generations: int = 10
    population_size: int = 100
    chunk_size: int = 256
    warm_start: bool = True
    reset_archive: bool = False
    target_mode: str = "overnight"
    custom_formulas: List[str] = []

@app.get("/health")
def health_check():
    return {
        "status": "ok",
        "service": "Bio-Quant Cloud GPU Worker",
        "device": DEVICE_NAME,
        "gpu_acceleration": USE_GPU,
        "vram_total_gb": VRAM_TOTAL_GB,
        "resources": HardwareArenaManager.telemetry()
    }

@app.post("/evolve")
def run_evolution_endpoint(req: EvolutionJobRequest):
    def event_stream():
        yield json.dumps({"event": "start", "ticker": req.ticker, "resources": HardwareArenaManager.telemetry()}) + "\n"
        bundle = get_or_fetch_market_buffers(req.ticker, lookback=req.lookback, target_mode=req.target_mode)
        n_len = bundle["n_len"]
        friction = 0.00012 if "BEES" in req.ticker else 0.00030
        archive = MapElitesArchive(20, 20)
        pop = [GeneticEngine.generate(max_depth=3) for _ in range(req.population_size)]
        pop.extend(GeneticEngine.warm_start_seeds())

        for gen in range(1, req.generations + 1):
            residents = archive.get_residents()
            parent_pool = [e.formula for e in residents] if len(residents) >= 4 else pop
            offspring = []
            while len(offspring) < req.population_size:
                p1, p2 = random.choice(parent_pool), random.choice(parent_pool)
                c1, c2 = GeneticEngine.crossover(p1, p2) if random.random() < 0.6 else (p1.clone(), p2.clone())
                offspring.extend([GeneticEngine.mutate(c1), GeneticEngine.mutate(c2)])
            offspring = offspring[:req.population_size]
            TensorASTEvaluator.clear_cache()
            inserted = archive.evaluate_batch(offspring, bundle["feature_tensors"], bundle["gross_rets_t"], n_len, friction, req.target_mode)
            yield json.dumps({
                "event": "generation", "gen": gen, "inserted": inserted,
                "total_elites": len(archive.get_residents()),
                "occupancy_pct": round(len(archive.get_residents()) / 4.0, 1),
                "leaderboard": archive.get_all_records()[:10]
            }) + "\n"

        yield json.dumps({"event": "complete", "total_elites": len(archive.get_residents()), "elite_records": archive.get_all_records()}) + "\n"
    return StreamingResponse(event_stream(), media_type="application/x-ndjson")

@app.get("/archive/{ticker}")
def get_archive(ticker: str):
    clean_t = ticker.replace(".NS", "").replace(".BO", "")
    archive = ACTIVE_ARCHIVES.get(clean_t, ACTIVE_ARCHIVES.get(ticker, MapElitesArchive(20, 20)))
    return {
        "status": "ok", "ticker": ticker,
        "resident_count": len(archive.get_residents()),
        "records": archive.get_all_records()
    }

print("Starting in-memory FastAPI server on 127.0.0.1:8000...")
config = uvicorn.Config(app=app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2.0)

print("Launching Cloudflare Tunnel...")
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "=" * 82)
    print("🟢 CLOUD GPU WORKER IS ONLINE & READY!")
    print(f"👉 PUBLIC TUNNEL URL: {tunnel_url}")
    print("Paste this into your Local WebUI (http://127.0.0.1:7860/) under Cloud GPU Acceleration!")
    print("=" * 82 + "\n")
else:
    print("Tunnel URL could not be resolved automatically. Check cloudflared logs.")
